# Chapter 2 Practical 04: Evaluation of Top-K Recommendations

Learning objectives:
- Compute Precision@K, Recall@K, and HitRate@K.
- Evaluate recommendations for multiple users.
- Read a compact metric table and bar chart.
- Distinguish accuracy, coverage, and hit-oriented satisfaction.

Slide connection: Precision@K, Recall@K, HitRate@K, ranking, and Top-N evaluation.


We use a tiny ground-truth example. In a real project, relevant items usually come from held-out ratings, clicks, purchases, or watch events.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

ground_truth = {
    "U1": {"The Martian", "Gravity", "The Matrix"},
    "U2": {"Toy Story", "Finding Nemo", "Paddington"},
    "U3": {"Titanic", "The Notebook", "La La Land"},
}

recommendations = {
    "U1": ["The Martian", "The Dark Knight", "Gravity", "Titanic", "Paddington"],
    "U2": ["Paddington", "Toy Story", "The Martian", "Finding Nemo", "Inception"],
    "U3": ["La La Land", "Titanic", "Interstellar", "The Notebook", "Toy Story"],
}

pd.DataFrame([
    {"user": u, "ground_truth": sorted(gt), "recommendations": recommendations[u]}
    for u, gt in ground_truth.items()
])


Precision@K asks: of the K recommended items, how many were relevant?


In [ ]:
def precision_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return hits / k

def recall_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return hits / len(relevant) if relevant else 0

def hitrate_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    return int(len(set(recommended_k) & set(relevant)) > 0)

precision_at_k(recommendations["U1"], ground_truth["U1"], k=3)


Now evaluate all users and display the results in one clean table.


In [ ]:
k = 3
rows = []
for user, relevant in ground_truth.items():
    recs = recommendations[user]
    rows.append({
        "user": user,
        f"Precision@{k}": precision_at_k(recs, relevant, k),
        f"Recall@{k}": recall_at_k(recs, relevant, k),
        f"HitRate@{k}": hitrate_at_k(recs, relevant, k),
        "hits_in_top_k": sorted(set(recs[:k]) & relevant),
    })

metrics = pd.DataFrame(rows)
metrics


Average the metrics across users to summarize system performance.


In [ ]:
summary = metrics[[f"Precision@{k}", f"Recall@{k}", f"HitRate@{k}"]].mean().to_frame("mean_value")
summary.round(3)


A simple bar chart makes it easier to compare the metric values.


In [ ]:
ax = summary.plot(kind="bar", legend=False, ylim=(0, 1), figsize=(6, 4))
ax.set_ylabel("Metric value")
ax.set_title(f"Average Top-{k} evaluation")
ax.bar_label(ax.containers[0], fmt="%.2f")
plt.xticks(rotation=0)
plt.show()


Accuracy, coverage, and satisfaction are related but not identical:

- Accuracy asks whether recommended items match known relevant items.
- Coverage asks whether the system can recommend a broad set of items instead of always the same few.
- HitRate@K asks whether the list contains at least one useful item for the user.

## What did we learn?

- Precision@K rewards short lists with many relevant items.
- Recall@K rewards finding a large share of all relevant items.
- HitRate@K is forgiving: one hit is enough.

Exercises:
1. Change `k` from 3 to 5. Which metric changes the most?
2. Add one more user with recommendations and ground truth.
